# Ablation Study — Attention Is All You Need

Każdy eksperyment używa małego modelu (d=256, 3 warstwy) trenowanego na 2000 zdaniach przez 10 epok.
Pozwala to na uruchomienie całości w ~2-4h na GPU.

| Eksperyment | Config | Co zmienia |
|---|---|---|
| Baseline | `ablations/abl_base` | punkt odniesienia |
| No PE | `ablations/abl_no_pos_enc` | `positional_encoding: none` |
| Single Head | `ablations/abl_single_head` | `h: 1` |
| Learned PE | `ablations/abl_learned_pe` | `positional_encoding: learned` |
| No Smoothing | `ablations/abl_no_smoothing` | `label_smoothing: 0.0` |
| Ckpt Averaging | — | bez treningu, tylko ewaluacja |

## 0. Setup

In [7]:
import json
import subprocess
import sys
from pathlib import Path

import torch

REPO = Path("__file__").resolve().parent.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.data.tokenizer import SharedBPETokenizer
from src.evaluation.generate import TranslationExample, generate_translations, load_model
from src.evaluation.metrics import compute_bleu

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# -- Tokenizer --
tokenizer = SharedBPETokenizer.load(
    REPO / "data/processed/bpe/bpe_vocab.json",
    REPO / "data/processed/bpe/bpe_merges.txt",
)
print(f"Tokenizer loaded, vocab_size={tokenizer.vocab_size}")

# -- Test data (first 500 for quick eval) --
def load_jsonl(path, max_samples=None):
    examples = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            rec = json.loads(line)["translation"]
            examples.append(TranslationExample(source=rec["de"], reference=rec["en"]))
    return examples

test_examples = load_jsonl(REPO / "data/raw/wmt14_de_en/test.jsonl", max_samples=500)
print(f"Loaded {len(test_examples)} test examples")

# -- Results accumulator --
results = {}

Device: cuda
Tokenizer loaded, vocab_size=32000
Loaded 500 test examples


## 1. Helper functions

In [8]:
def run_ablation(config_name: str, extra_overrides: list[str] | None = None) -> dict:
    """Train with an ablation config composed into the global Hydra namespace."""
    extra_overrides = extra_overrides or []
    short_name = config_name.split("/", 1)[-1]

    # Preferred invocation: compose train + ablation at global package level.
    cmd_variants = [
        [
            sys.executable,
            str(REPO / "train.py"),
            "--config-name=train",
            f"+ablations@_global_={short_name}",
            *extra_overrides,
        ],
        [
            sys.executable,
            str(REPO / "train.py"),
            f"--config-name={config_name}",
            *extra_overrides,
        ],
    ]

    last_returncode = 1
    for i, cmd in enumerate(cmd_variants, start=1):
        print(f"\n>>> Running (variant {i}): {' '.join(cmd)}")
        result = subprocess.run(cmd, cwd=REPO, capture_output=False, text=True)
        last_returncode = result.returncode
        if result.returncode == 0:
            return {"config": config_name, "returncode": 0, "variant": i}

    print(f"[WARN] Training failed for all command variants: {config_name}")
    return {"config": config_name, "returncode": last_returncode, "variant": None}


def quick_bleu(
    checkpoint_path: str | Path,
    examples: list,
    cfg_overrides: dict | None = None,
    beam_size: int = 4,
    max_len: int = 64,
) -> float:
    """Load checkpoint and score BLEU on provided examples."""
    cfg_defaults = dict(
        vocab_size=32000,
        d_model=256,
        num_layers=3,
        h=4,
        d_ff=512,
        dropout=0.0,
        max_len=256,
        positional_encoding="sinusoidal",
    )
    if cfg_overrides:
        cfg_defaults.update(cfg_overrides)

    from src.models.transformer import Transformer
    from src.evaluation.generate import extract_model_state

    model = Transformer(**cfg_defaults)
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(extract_model_state(ckpt))
    model.eval()

    hypotheses, references = generate_translations(
        model=model,
        tokenizer=tokenizer,
        examples=examples,
        batch_size=16,
        beam_size=beam_size,
        max_len=max_len,
        device=DEVICE,
    )
    score = compute_bleu(hypotheses, references) if references else float("nan")
    print(f"  BLEU: {score:.2f}")
    return score


def last_checkpoint(ckpt_dir: str | Path) -> Path | None:
    """Return the latest epoch or step checkpoint in a directory."""
    candidates = sorted(
        Path(ckpt_dir).glob("*.pt"),
        key=lambda p: p.stat().st_mtime,
    )
    return candidates[-1] if candidates else None


## 2. Baseline
Mały model (d=256, 3 warstwy, sinusoidal PE, label_smoothing=0.1) — punkt odniesienia dla wszystkich ablacji.

**Oczekiwane:** ~2-5 BLEU na 500 zdaniach (mały model, 10 epok, 2000 zdań treningowych).

In [9]:
run_ablation("ablations/abl_base")
ckpt = last_checkpoint(REPO / "checkpoints/ablations/baseline")
bleu = quick_bleu(ckpt, test_examples) if ckpt else float("nan")
results["baseline"] = {"bleu": bleu, "checkpoint": str(ckpt)}
print(f"Baseline BLEU: {bleu:.2f}")


>>> Running (variant 1): /home/olek/github_projekty/Attention_Transformer_Reproduction/.venv/bin/python /home/olek/github_projekty/Attention_Transformer_Reproduction/train.py --config-name=train +ablations@_global_=abl_base
Model: 28,561,664 params  device=cuda  smoke=False


epoch 1/10:  40%|████      | 50/125 [00:02<00:02, 26.22batch/s, grad_norm=1.520, loss=6.4668, lr=3.98e-04, skipped=0]

epoch=1 step=50 loss=6.650576 grad_norm=1.5894 lr=3.91e-04


epoch 1/10:  81%|████████  | 101/125 [00:04<00:00, 25.60batch/s, grad_norm=0.864, loss=5.7096, lr=7.97e-04, skipped=0]

epoch=1 step=100 loss=5.625921 grad_norm=0.6329 lr=7.81e-04


...

epoch 10/10:  79%|███████▉  | 99/125 [00:03<00:00, 26.02batch/s, grad_norm=0.285, loss=5.5135, lr=1.79e-03, skipped=0]

epoch=10 step=1225 loss=5.513490 grad_norm=0.2849 lr=1.79e-03


epoch=10 train_loss=5.625011 finite=125 skipped=0
epoch=10 val_loss=6.786736
saved checkpoint: checkpoints/ablations/baseline/epoch_010.pt


generate: 100%|██████████| 32/32 [01:13<00:00,  2.31s/batch]

  BLEU: 0.01
Baseline BLEU: 0.01


## 3. Eksperyment 1 — No Positional Encoding
Model widzi tokeny jako nieuporządkowany zbiór — brak informacji o pozycji w sekwencji.

**Oczekiwane:** znaczący spadek BLEU (~5-10 pkt), model radzi sobie gorzej z długimi zdaniami i słownikiem zależnym od kolejności.

In [10]:
run_ablation("ablations/abl_no_pos_enc")
ckpt = last_checkpoint(REPO / "checkpoints/ablations/no_pos_enc")
bleu = quick_bleu(ckpt, test_examples, cfg_overrides={"positional_encoding": "none"}) if ckpt else float("nan")
results["no_pos_enc"] = {"bleu": bleu, "delta": bleu - results["baseline"]["bleu"]}
print(f"No PE BLEU: {bleu:.2f}  (delta: {results['no_pos_enc']['delta']:+.2f})")


>>> Running (variant 1): /home/olek/github_projekty/Attention_Transformer_Reproduction/.venv/bin/python /home/olek/github_projekty/Attention_Transformer_Reproduction/train.py --config-name=train +ablations@_global_=abl_no_pos_enc
Model: 28,561,664 params  device=cuda  smoke=False


epoch 1/10:  40%|████      | 50/125 [00:02<00:02, 26.67batch/s, grad_norm=1.520, loss=6.4704, lr=3.98e-04, skipped=0]

epoch=1 step=50 loss=6.653882 grad_norm=1.5901 lr=3.91e-04


epoch 1/10:  81%|████████  | 101/125 [00:04<00:00, 25.55batch/s, grad_norm=0.829, loss=5.8181, lr=7.89e-04, skipped=0]

epoch=1 step=100 loss=5.626289 grad_norm=0.6346 lr=7.81e-04


...

epoch 10/10:  82%|████████▏ | 102/125 [00:04<00:00, 25.57batch/s, grad_norm=0.392, loss=5.7492, lr=1.78e-03, skipped=0]

epoch=10 step=1225 loss=5.510616 grad_norm=0.4024 lr=1.79e-03


epoch=10 train_loss=5.624672 finite=125 skipped=0
epoch=10 val_loss=6.764601
saved checkpoint: checkpoints/ablations/no_pos_enc/epoch_010.pt


generate: 100%|██████████| 32/32 [01:13<00:00,  2.30s/batch]

  BLEU: 0.00
No PE BLEU: 0.00  (delta: -0.01)


## 4. Eksperyment 2 — Single Head Attention
Jeden head zamiast 4 — model nie może równolegle śledzić różnych typów zależności.

**Oczekiwane:** wolniejsza konwergencja, spadek BLEU ~2-5 pkt.

In [11]:
run_ablation("ablations/abl_single_head")
ckpt = last_checkpoint(REPO / "checkpoints/ablations/single_head")
bleu = quick_bleu(ckpt, test_examples, cfg_overrides={"h": 1}) if ckpt else float("nan")
results["single_head"] = {"bleu": bleu, "delta": bleu - results["baseline"]["bleu"]}
print(f"Single head BLEU: {bleu:.2f}  (delta: {results['single_head']['delta']:+.2f})")


>>> Running (variant 1): /home/olek/github_projekty/Attention_Transformer_Reproduction/.venv/bin/python /home/olek/github_projekty/Attention_Transformer_Reproduction/train.py --config-name=train +ablations@_global_=abl_single_head
Model: 28,561,664 params  device=cuda  smoke=False


epoch 1/12:  42%|████▏     | 53/125 [00:02<00:02, 27.51batch/s, grad_norm=1.478, loss=6.3852, lr=4.22e-04, skipped=0]

epoch=1 step=50 loss=6.647553 grad_norm=1.5898 lr=3.91e-04


epoch 1/12:  78%|███████▊  | 98/125 [00:04<00:01, 26.06batch/s, grad_norm=0.631, loss=5.6236, lr=7.81e-04, skipped=0]

epoch=1 step=100 loss=5.623559 grad_norm=0.6310 lr=7.81e-04


...

epoch 12/12:  81%|████████  | 101/125 [00:03<00:00, 26.27batch/s, grad_norm=0.460, loss=5.5306, lr=1.63e-03, skipped=0]

epoch=12 step=1475 loss=5.566053 grad_norm=0.4768 lr=1.63e-03


epoch=12 train_loss=5.565278 finite=125 skipped=0
epoch=12 val_loss=7.151651


generate: 100%|██████████| 32/32 [01:10<00:00,  2.21s/batch]

  BLEU: 0.01
Single head BLEU: 0.01  (delta: -0.00)


## 5. Eksperyment 3 — Learned Positional Encoding
Pozycje uczone zamiast sinusoidalnych — model sam odkrywa reprezentacje pozycji.

**Oczekiwane:** podobny BLEU ±1 pkt względem baseline. Learned PE może działać gorzej na sekwencjach dłuższych niż te widziane w treningu.

In [12]:
run_ablation("ablations/abl_learned_pe")
ckpt = last_checkpoint(REPO / "checkpoints/ablations/learned_pe")
bleu = quick_bleu(ckpt, test_examples, cfg_overrides={"positional_encoding": "learned"}) if ckpt else float("nan")
results["learned_pe"] = {"bleu": bleu, "delta": bleu - results["baseline"]["bleu"]}
print(f"Learned PE BLEU: {bleu:.2f}  (delta: {results['learned_pe']['delta']:+.2f})")


>>> Running (variant 1): /home/olek/github_projekty/Attention_Transformer_Reproduction/.venv/bin/python /home/olek/github_projekty/Attention_Transformer_Reproduction/train.py --config-name=train +ablations@_global_=abl_learned_pe
Model: 28,692,736 params  device=cuda  smoke=False


epoch 1/10:  41%|████      | 51/125 [00:02<00:02, 25.22batch/s, grad_norm=1.718, loss=6.3574, lr=4.14e-04, skipped=0]

epoch=1 step=50 loss=6.602783 grad_norm=1.6716 lr=3.91e-04


epoch 1/10:  79%|███████▉  | 99/125 [00:04<00:00, 26.12batch/s, grad_norm=0.722, loss=5.6564, lr=7.81e-04, skipped=0]

epoch=1 step=100 loss=5.656416 grad_norm=0.7215 lr=7.81e-04


...

epoch 10/10:  82%|████████▏ | 102/125 [00:04<00:00, 25.47batch/s, grad_norm=1.091, loss=5.5868, lr=1.78e-03, skipped=0]

epoch=10 step=1225 loss=5.462485 grad_norm=0.7139 lr=1.79e-03


epoch=10 train_loss=5.526106 finite=125 skipped=0
epoch=10 val_loss=7.893127
saved checkpoint: checkpoints/ablations/learned_pe/epoch_010.pt


generate: 100%|██████████| 32/32 [01:12<00:00,  2.26s/batch]

  BLEU: 0.00
Learned PE BLEU: 0.00  (delta: -0.00)


## 6. Eksperyment 4 — No Label Smoothing
Model optymalizuje hard one-hot targets zamiast rozkładu wygładzonego.

**Oczekiwane:** wyższe perplexity, niższy BLEU — model overconfident, gorzej generalizuje.

In [13]:
run_ablation("ablations/abl_no_smoothing")
ckpt = last_checkpoint(REPO / "checkpoints/ablations/no_smoothing")
bleu = quick_bleu(ckpt, test_examples) if ckpt else float("nan")
results["no_smoothing"] = {"bleu": bleu, "delta": bleu - results["baseline"]["bleu"]}
print(f"No smoothing BLEU: {bleu:.2f}  (delta: {results['no_smoothing']['delta']:+.2f})")


>>> Running (variant 1): /home/olek/github_projekty/Attention_Transformer_Reproduction/.venv/bin/python /home/olek/github_projekty/Attention_Transformer_Reproduction/train.py --config-name=train +ablations@_global_=abl_no_smoothing
Model: 28,561,664 params  device=cuda  smoke=False


epoch 1/10:  39%|███▉      | 49/125 [00:02<00:02, 26.93batch/s, grad_norm=1.703, loss=7.4964, lr=3.98e-04, skipped=0] 

epoch=1 step=50 loss=7.704494 grad_norm=1.7780 lr=3.91e-04


epoch 1/10:  81%|████████  | 101/125 [00:04<00:00, 25.70batch/s, grad_norm=0.912, loss=6.6145, lr=7.89e-04, skipped=0]

epoch=1 step=100 loss=6.384234 grad_norm=0.7052 lr=7.81e-04


...

epoch 10/10:  79%|███████▉  | 99/125 [00:03<00:01, 25.78batch/s, grad_norm=0.579, loss=6.0395, lr=1.79e-03, skipped=0]  

epoch=10 step=1225 loss=6.004796 grad_norm=5.6590 lr=1.79e-03


epoch=10 train_loss=6.185566 finite=125 skipped=0
epoch=10 val_loss=10.593152
saved checkpoint: checkpoints/ablations/no_smoothing/epoch_010.pt


generate: 100%|██████████| 32/32 [01:09<00:00,  2.18s/batch]

  BLEU: 0.00
No smoothing BLEU: 0.00  (delta: -0.00)


## 7. Eksperyment 5 — Checkpoint Averaging (Base EN-DE)
Porównanie pełnego modelu bazowego: pojedynczy checkpoint vs uśredniony.
**Bez treningu** — używa istniejących checkpointów z `checkpoints/base_en_de`.

**Oczekiwane:** averaged powinien dać +0.5–2 BLEU dzięki redukcji szumu w wagach.

In [14]:
# Konfiguracja pełnego modelu bazowego (512d, 6 warstw, 8 głów)
BASE_CFG = dict(vocab_size=32000, d_model=512, num_layers=6, h=8, d_ff=2048,
                dropout=0.0, max_len=1024, positional_encoding="sinusoidal")

averaged_ckpt = REPO / "checkpoints/base_en_de/averaged.pt"
single_ckpt   = REPO / "checkpoints/base_en_de_v2/step_039062.pt"

bleu_avg    = quick_bleu(averaged_ckpt, test_examples, cfg_overrides=BASE_CFG) if averaged_ckpt.exists() else float("nan")
bleu_single = quick_bleu(single_ckpt,   test_examples, cfg_overrides=BASE_CFG) if single_ckpt.exists()   else float("nan")

results["base_averaged"] = {"bleu": bleu_avg}
results["base_single"]   = {"bleu": bleu_single}
print(f"Averaged: {bleu_avg:.2f}  |  Single: {bleu_single:.2f}  |  Delta: {bleu_avg - bleu_single:+.2f}")

generate: 100%|██████████| 32/32 [02:26<00:00,  4.57s/batch]
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


  BLEU: 5.72


generate: 100%|██████████| 32/32 [02:29<00:00,  4.66s/batch]
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


  BLEU: 3.51
Averaged: 5.72  |  Single: 3.51  |  Delta: +2.21


## 8. Podsumowanie wyników

In [15]:
import pandas as pd

baseline_bleu = results.get("baseline", {}).get("bleu", float("nan"))
rows = []
for name, data in results.items():
    bleu = data.get("bleu", float("nan"))
    delta = bleu - baseline_bleu if name not in ("baseline", "base_averaged", "base_single") else None
    rows.append({"Eksperyment": name, "BLEU": round(bleu, 2),
                 "Delta vs baseline": f"{delta:+.2f}" if delta is not None else "—"})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

  Eksperyment  BLEU Delta vs baseline
     baseline  0.01                 —
   no_pos_enc  0.00             -0.01
  single_head  0.01             -0.00
   learned_pe  0.00             -0.00
 no_smoothing  0.00             -0.00
base_averaged  5.72                 —
  base_single  3.51                 —
